## Εισαγωγή βιβλιοθηκων 

In [1]:
# If using Colab Notebook, install faiss and transformers right in the notebook
!pip install faiss-cpu
!pip install transformers
!pip install torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import sys
print(sys.executable)
!"{sys.executable}" -m pip install faiss-cpu
!"{sys.executable}" -m pip install torch
!"{sys.executable}" -m pip install transformers
!"{sys.executable}" -m pip install -U sentence-transformers tqdm


C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.3 kB ? eta -:--:--
   ------------------ --------------------- 262.1/571.3 kB ? eta -:--:--
   ---------------------------------- --- 524.3/571.3 kB 310.6 kB/s eta 0:00:01
   ---------------------------------------- 571.3/571.3 kB 291.1 kB/s  0:00:01

  Attempting uninstall: tqdm

    Found existing installation: tqdm 4.67.1

    Uninstalling tqdm-4.67.1:

      Successfully uninstalled tqdm-4.67.1

   ---------------------------------------- 0/2 [tqdm]
   ---------------------------------------- 0/2 [tqdm]
   ---------------------------------------- 0/2 [tqdm]
   ----------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\ioann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import csv
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

## Προεπεξεργασία του CSV

In [2]:
all_data = []
#διαβάζει το csv σαν dictionary ανά γραμμή 
with open("IR2025/documents.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        all_data.append(row)

document_texts = [] # λίστα με τα κείμενα των εγγραφών
document_ids = [] # λίστα με τα id's των εγγραφών

for doc in all_data:
    text = doc["Text"].strip().replace("\n", " ").replace("\t"," ") # αφαιρεί κενά από την αρχη και το τέλος/ αντικαθιστά τις αλλαγές γραμμής με το κενό/ αντικαθιστά τα tabs με κενό
    document_texts.append(text) # προσθέτει στην λίστα το κείμενο
    document_ids.append(int(doc["ID"])) # προσθέτει το id στην λίστα σε μορφή int

print(f"length of documents: {len(document_texts)}") # αριθμός εγγγράφων που διάβασε
print(f"First document: {document_texts[0][:200]}...") # εμφανίζει τους πρώτους 200 χαρακτήρες

length of documents: 18316
First document: Support towards the Europe PMC initiative-Contribution for 2014-2016: "The proposed action will provide continued support to the European Research Council (ERC) in the implementation of its Open Acces...


## Φορτώνουμε το μοντέλο sentence-transformers

In [3]:
# φορτώνει το προεκπαιδευμένο μοντέλο και παράγει embeddings 384 διαστάσεων
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
# Υπολογίζει embeddings για όλα τα κείμενα 
embeddings = model.encode(document_texts, show_progress_bar=True, batch_size=256, convert_to_numpy=True)# τα επεξεργάζεται σε παρτίδες (batch_size=128)/ επιστρέφει numpy για την faiss
#κανονικοποιεί τα embeddings σε διάνυσμα μήκους 1 
faiss.normalize_L2(embeddings)
#εκτυπώνει αριθμό κειμενων και την διάσταση
print(f"Embeddings shape: {embeddings.shape}")


Batches:   0%|          | 0/72 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Δημιουργούμε FAISS Index (IndexFlatIP για Cosine Similirity)

In [7]:
# διάσταση των embeddings
dimension = embeddings.shape[1]
# φτιάχνει το index του faiss
index = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
# προσθέτει τα embeddings μαζί με τα document ids 
index.add_with_ids(embeddings, np.array(document_ids))
# εμφανίζει πόσα διανύσματα έχουν αποθηκευτεί στο index
print(f"Number of vectors in FAISS index: {index.ntotal}")

Number of vectors in FAISS index: 18316


## Αναζήτηση Queries 

In [8]:
# Συνάρτηση που παίρνει το κείμενο του ερωτήματος και τα αποτελέσματα επιστροφής
def search(query, k):
    # Υπολογίζει embedding query
    query_vec = model.encode([query], convert_to_numpy=True)
    # κανινικοποίηση του query ώστε να έχει μήκος 1 
    faiss.normalize_L2(query_vec)
    # Αναζητάει top-k
    scores, ids = index.search(query_vec, k)
    # Επιστρέφει λίστα (iD, score, text)
    results = []
    for idx,score in zip(ids[0], scores[0]):
        doc_text = next(doc["Text"] for doc in all_data if int(doc["ID"]) == idx)
        results.append((idx, score, doc_text[:200]+"..."))  # εμφανίζουμε πρώτα 200 chars
    return results


## Παραδειγμα Αναζητησης

In [9]:
query = "spanish flu casualties"
top_results = search(query, k=5)

for r in top_results:
    print(f"ID: {r[0]} | Score: {r[1]:.4f} | Text: {r[2]}")


ID: 195790 | Score: 0.4213 | Text: Quantiative Study of Major Historic Epidemics and Transitions to longer, healthier lives: Only 150 years ago, one in five Europeans died in infancy, life expectancy was 40 years, and the leading cause...
ID: 204417 | Score: 0.3785 | Text: FLUSENSOR - innovative, ultra-sensitive, fast and cheap micro-test to detect influenza virus: Influenza is one of the most common viral infectious diseases and a significant source of incidence and de...
ID: 215465 | Score: 0.3607 | Text: Tracing the inFLUenza vaccine imPRINT on immune system to identify cellular signature of protection: Influenza virus causes a large socioeconomic burden on society, with 70 000 deaths every year in Eu...
ID: 212807 | Score: 0.3559 | Text: Disasters, Communication and Politics in South-Western Europe: the Making of Emergency Response Policies in the Early Modern Age: The connections between the circulation of news of extreme events, the...
ID: 196375 | Score: 0.3547 | Text: Developme

## Φορτώνουμε τα Queries

In [21]:
import csv 

queries_data = []
with open("IR2025/queries.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        queries_data.append(row)

query_ids=[]
query_texts =[]

for q in queries_data:
    text = q["Text"].strip().replace("\n", " ").replace("\t", " ")
    query_texts.append(text)
    query_ids.append((q["ID"]).strip())

print(f"lenght of queries : {len(query_texts)}")
print(f"First query : {query_texts[0]}")



lenght of queries : 10
First query : EUTRAVEL Optimodal European Travel Ecosystem EuTravel aims to:\n1. Support the EU agenda towards an open and single market for mobility services by enabling travellers to organise a multimodal trip in accordance with their own criteria including environmental performance, providing multimodal travel service providers an effective way to deliver customised services addressing any type of specialised travel needs and facilitating fact-based EU policy making.\n2. Promote the creation of content, open and linked data for travellers enriching the travelling experience.\n3. Support travel industry players join forces towards realising an EU shared seamless mobility strategy and architecture.\nEuTravel will research and demonstrate Inter-modal travel optimised with respect to synchronisation between modes, passenger experience and rights and environmental performance (Optimodal Travel). \n\nThe project objectives will be realised by:\n1. Developing an open

In [ ]:
import os 

doc_embedding = "doc_embedding.npy"

if os.path.exists (doc_embedding) :
    embeddings = np.load(doc_embedding)
else:
    query_embeddings = model.encode(
        query_texts,
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    faiss.normalize_L2(query_embeddings)
    np.save(doc_embedding,embeddings)

print("Query embeddings shape:", query_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query embeddings shape: (10, 384)


## Run Files για όλα τα Queries

In [23]:
def build_run_file(file_name, k, system_name="semantic-faiss"):
    with open(file_name, "w", encoding="utf-8") as f:
        for qid, qvec in zip(query_ids, query_embeddings):
            qvec = qvec.reshape(1, -1).astype("float32")

            scores, ids = index.search(qvec, k)

            for rank, (doc_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
                if doc_id == -1:
                    continue
                f.write(f"{qid} Q0 {int(doc_id)} {rank} {float(score):.6f} {system_name}\n")

build_run_file("run_k20.txt", 20)
build_run_file("run_k30.txt", 30)
build_run_file("run_k50.txt", 50)

print("Run files created")


Run files created
